# Deploying NVIDIA Nemotron-3-Nano-Omni with TensorRT LLM

This notebook will walk you through how to run the `nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning` model via TensorRT-LLM.

[TensorRT LLM](https://nvidia.github.io/TensorRT-LLM/) is NVIDIA’s open-source library for accelerating and optimizing LLM inference performance on NVIDIA GPUs. 

For more details on the model [click here](https://huggingface.co/nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16). 

Prerequisites:
- NVIDIA GPU with recent drivers (≥ 64 GB VRAM for BF16, ≥ 32 GB for FP8, ≥ 20 GB for NVFP4) and CUDA 13.0+
- Python 3.10+
- TensorRT-LLM (you can refer to NVIDIA documentation, or pull this [container](https://catalog.ngc.nvidia.com/orgs/nvidia/teams/tensorrt-llm/containers/release?version=1.3.0rc13))

#### Launch on NVIDIA Brev
You can simplify the environment setup by using [NVIDIA Brev](https://developer.nvidia.com/brev). Click the button to launch this project on a Brev instance with the necessary dependencies pre-configured.

Once deployed, click on the "Open Notebook" button to get started with this guide. 

**For H100 (BF16/FP8 models):**

[![Launch on Brev](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/launchable/deploy?launchableID=env-3Cm3y3jUbb1Q28MPXEQLTG7R61m)

**For RTX PRO 6000 (NVFP4 model - requires Blackwell architecture):**

[![Launch on Brev](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/launchable/deploy?launchableID=env-3Cm43kXGhTHVccjTjEqcRyekhLi)


## Prerequisites & environment

Set up a containerized environment for TensorRT-LLM by running the following in a **host** terminal.

For example, on Brev you can open a terminal directly. If you are not on Brev, use any terminal with Docker and GPU access.

```shell
docker run --rm -it --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 --gpus=all -p 8000:8000 nvcr.io/nvidia/tensorrt-llm/release:1.3.0rc13
```

You now have TRT-LLM set up!

In [1]:
#If pip not found
!python -m ensurepip --default-pip

Looking in links: /tmp/tmp5wayaclc
Processing /tmp/tmp5wayaclc/pip-25.0.1-py3-none-any.whl


In [4]:
%pip install torch openai numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 9.4 MB/s eta 0:00:000:00:01

[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Verify GPU

Check that CUDA is available and the GPU is detected correctly.


In [5]:
# Environment check
import sys
import torch

print(f"Python: {sys.version}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Num GPUs: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU[{i}]: {torch.cuda.get_device_name(i)}")


Python: 3.12.13 (main, Apr 14 2026, 14:29:00) [Clang 22.1.3 ]
CUDA available: True
Num GPUs: 1
GPU[0]: NVIDIA H100 80GB HBM3


## OpenAI-compatible server

Start a local OpenAI-compatible server with TensorRT-LLM via the terminal, within the running docker container.

Ensure that the following commands are executed from the docker terminal.

### Create a YAML file with the required configuration

```shell
cat > nano_v3.yaml<<EOF
kv_cache_config:
  enable_block_reuse: false
  free_gpu_memory_fraction: 0.80
  mamba_ssm_cache_dtype: float32
max_batch_size: 128
EOF
```

> **Notes:**
> - If you wish to enable chunked prefill (via `enable_chunked_prefill: true`), only do so if you do **not** plan to have video-based workloads. Support for videos and chunked prefill for this model is a work-in-progress.
> - If your workload has long audio / video inputs at high concurrency, on more memory-constrained SKUs, you may need to lower the `max_batch_size` or `free_gpu_memory_fraction` to accommodate their inputs / outputs, and / or enable chunked prefill (see previous bullet point).
> - For NVFP4 on B200 specifically, the MoE backend can be changed as follows for better performance: `moe_config: backend: TRTLLM`


### Audio from video

> **Notes:**
> - the usage of PyAV means that the TensorRT-LLM runtime environment cannot be redistributed in any form, as that would violate the EULAs of NVIDIA's proprietary software.
> - there is a known limitation where the first request has to be a video (as opposed to having other modalities). A fix is being worked on.

This model supports extracting audio from videos. In order to do so, the [PyAV](https://github.com/pyav-org/pyav) python package needs to be installed, and the `TRTLLM_ENABLE_PYAV=1` environment variable set.

This feature can then be enabled at serving time by supplying `--media_io_kwargs '{"video": {"extract_audio": true}}'` to the `trtllm-serve` command as well.


### Load the model

#### BF16 version

```shell
PYTORCH_ALLOC_CONF=expandable_segments:True \
trtllm-serve serve "nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16" \
--host 0.0.0.0 \
--port 8000 \
--trust_remote_code \
--reasoning_parser nano-v3 \
--tool_parser qwen3_coder \
--extra_llm_api_options nano_v3.yaml
```

#### Alternative: Load the FP8 quantized version for faster inference and lower memory usage

```shell
PYTORCH_ALLOC_CONF=expandable_segments:True \
trtllm-serve serve "nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-FP8" \
--host 0.0.0.0 \
--port 8000 \
--trust_remote_code \
--reasoning_parser nano-v3 \
--tool_parser qwen3_coder \
--extra_llm_api_options nano_v3.yaml
```

### Alternative: Load the NVFP4 quantized version for high efficiency and lowest memory usage

```shell
PYTORCH_ALLOC_CONF=expandable_segments:True \
trtllm-serve serve "nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-NVFP4" \
--host 0.0.0.0 \
--port 8000 \
--trust_remote_code \
--reasoning_parser nano-v3 \
--tool_parser qwen3_coder \
--extra_llm_api_options nano_v3.yaml
```

### DGX Spark

For DGX Spark, it’s recommended to lower the max_batch_size due to the limited memory capacity.

```shell
cat > nano_v3.yaml<<EOF
kv_cache_config:
  enable_block_reuse: false
  free_gpu_memory_fraction: 0.80
  mamba_ssm_cache_dtype: float32
moe_config:
   backend: CUTLASS
cuda_graph_config:
    enable_padding: true
    max_batch_size: 1
max_batch_size: 1
EOF
```

And then load the NVFP4 quantized version for high efficiency and lowest memory usage

```shell
PYTORCH_ALLOC_CONF=expandable_segments:True \
trtllm-serve serve "nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-NVFP4" \
--host 0.0.0.0 \
--port 8000 \
--trust_remote_code \
--reasoning_parser nano-v3 \
--tool_parser qwen3_coder \
--extra_llm_api_options nano_v3.yaml
```

Your server is now running!

### Use the API

Use the OpenAI-compatible client to send requests to the TensorRT-LLM server.

Note: The model supports two modes - Reasoning ON (default) vs OFF. This can be toggled by setting enable_thinking to False, as shown below. 

In [6]:
from openai import OpenAI
import requests

# Setup client
BASE_URL = "http://0.0.0.0:8000/v1"
API_KEY = "null" 
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

model_id = "nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16" # set this to the model you loaded

In [11]:
# Reasoning on (default)
print("Reasoning on")
response = client.chat.completions.create(
    model=model_id,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs"}
    ],
    temperature=1,
    max_tokens=512,
)
print(response.choices[0].message.reasoning_content, response.choices[0].message.content)

print("\n")

# Reasoning off
print("Reasoning off")
response = client.chat.completions.create(
    model=model_id,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Give me 3 bullet points about TensorRT-LLM."}
    ],
    temperature=0,
    max_tokens=256,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}}
)
print(response.choices[0].message.reasoning_content, response.choices[0].message.content)

Reasoning on
We need to respond with a haiku about GPUs. Haiku is 5-7-5 syllable structure, typically about nature, but can be any. Must be correct syllable count. Let's craft a haiku: "Silent cores glow" (5? let's count: Si-lent (2) cores (1) glow (1) =4). Need 5 syllables. Maybe "Silent cores softly glow" Count: Si-lent(2) cores(1) soft-ly(2) glow(1) =6. Too many. Let's think.

First line 5 syllables:
"Metal hearts ignite" Count: Met-al (2) hearts (1) ig-nite (2) =5. Good.

Second line 7 syllables:
"Parallel streams race through light" Count: Par-al (2) streams (1) race (1) through (1) light (1) =6? Wait count: Par(1) al(2) =2, streams(3) race(4) through(5) light(6). That's 6. Need 7. Add "bright" maybe. "Parallel streams race through bright light". Count: Par(1) al(2) streams(3) race(4) through(5) bright(6) light(7). Good 7.

Third line 5 syllables:
"Dreams in silicon" Count: Dreams (1) in (1) si-licon (2) =4. Need 5. Maybe "Dreams in silicon bloom". Dreams(1) in(2) si(3) lon(4) blo

In [7]:
# Streaming chat completion
print("Streaming response:")
stream = client.chat.completions.create(
    model=model_id,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What are the first 5 prime numbers?"}
    ],
    temperature=0.7,
    max_tokens=1024,
    stream=True,
)

for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

Streaming response:

The first five prime numbers are:

**2, 3, 5, 7, 11**.

### Tool calling

Use the OpenAI tools schema to call functions via the TensorRT-LLM endpoint.

In [8]:
# Tool calling via OpenAI tools schema
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate_tip",
            "parameters": {
                "type": "object",
                "properties": {
                    "bill_total": {
                        "type": "integer",
                        "description": "The total amount of the bill"
                    },
                    "tip_percentage": {
                        "type": "integer",
                        "description": "The percentage of tip to be applied"
                    }
                },
                "required": ["bill_total", "tip_percentage"]
            }
        }
    }
]

completion = client.chat.completions.create(
    model=model_id,
    messages=[
        {"role": "system", "content": ""},
        {"role": "user", "content": "My bill is $50. What will be the amount for 15% tip?"}
    ],
    tools=TOOLS,
    temperature=0.6,
    top_p=0.95,
    max_tokens=512,
    stream=False
)

print(completion.choices[0].message.reasoning_content, completion.choices[0].message.content)
print(completion.choices[0].message.tool_calls)

Okay, the user wants to calculate a 15% tip on a $50 bill. Let me see. The tool provided is calculate_tip, which requires bill_total and tip_percentage. The bill_total is $50, so as an integer that's 50. The tip percentage is 15, so I need to plug those into the function. Let me check if there are any required parameters. The tool's required fields are both bill_total and tip_percentage, which the user provided. So I should call calculate_tip with 50 and 15. The function should return the tip amount, which is 50 multiplied by 0.15, resulting in $7.50. I need to make sure the JSON is correctly formatted with the arguments as integers.
 


[ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-9bdee2d401a8466e84d2654e7a5786ee', function=Function(arguments='{"bill_total": 50, "tip_percentage": 15}', name='calculate_tip'), type='function')]


### Video understanding

Nemotron-3-Nano-Omni is a multimodal model that can reason over text, images, audio, and video.

The cell below demonstrates video understanding: it sends a short NVIDIA Omniverse clip to the model via the OpenAI-compatible API and asks it to describe the scene.

In [9]:
video_url = "https://blogs.nvidia.com/wp-content/uploads/2023/04/nvidia-studio-itns-wk53-scene-in-omniverse-1280w.mp4"

resp = client.chat.completions.create(
    model=model_id,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe what happens in this video."},
                {"type": "video_url", "video_url": {"url": video_url}},
            ],
        }
    ],
    temperature=0.6,
    max_tokens=2048,
)

print(resp.choices[0].message.content)


The video displays a screen recording of a 3D modeling software interface, likely Blender, focusing on a detailed winter scene.

*   **00:00 - 00:05**: The camera starts with a wider shot of a small, circular island covered in snow, surrounded by water. A small wooden cabin sits in the center, surrounded by tall pine trees. The camera slowly zooms in and begins to orbit around the scene.
*   **00:05 - 00:10**: As the camera gets closer, more details of the environment become visible. There is a wooden fence enclosing part of the yard, scattered logs, and a barrel. A figure, appearing to be a person or perhaps an animal, is standing near a table outside the cabin.
*   **00:10 - 00:12**: The camera continues to move closer and pan slightly, offering a clearer view of the cabin's exterior, which has a snow-covered roof and wooden siding. The lighting highlights the textures of the snow and the surrounding forest floor. The software panels on the right side of the screen remain visible th

### Controlling Reasoning Budget

The `reasoning_budget` parameter allows you to limit the length of the model's reasoning trace. When the reasoning output reaches the specified token budget, the model will attempt to gracefully end the reasoning at the next newline character. 

If no newline is encountered within 500 tokens after reaching the budget threshold, the reasoning trace will be forcibly terminated at `reasoning_budget + 500` tokens to prevent excessive generation.


In [ ]:
from typing import Any, Dict, List
import openai
from transformers import AutoTokenizer


class ThinkingBudgetClient:
    def __init__(self, base_url: str, api_key: str, tokenizer_name_or_path: str):
        self.base_url = base_url
        self.api_key = api_key
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name_or_path)
        self.client = openai.OpenAI(base_url=self.base_url, api_key=self.api_key)

    def chat_completion(
        self,
        model: str,
        messages: List[Dict[str, Any]],
        reasoning_budget: int = 512,
        max_tokens: int = 1024,
        **kwargs,
    ) -> Dict[str, Any]:
        assert (
            max_tokens > reasoning_budget
        ), f"reasoning_budget must be smaller than max_tokens. Given {max_tokens=} and {reasoning_budget=}"

        # 1. first call chat completion to get reasoning content
        response = self.client.chat.completions.create(
            model=model, 
            messages=messages, 
            max_tokens=reasoning_budget, 
            **kwargs
        )
        
        reasoning_content = response.choices[0].message.reasoning_content or ""
        
        if "</think>" not in reasoning_content:
            # reasoning content is too long, closed with a period (.)
            reasoning_content = f"{reasoning_content}.\n</think>\n\n"
        
        reasoning_tokens_used = len(
            self.tokenizer.encode(reasoning_content, add_special_tokens=False)
        )
        remaining_tokens = max_tokens - reasoning_tokens_used
        
        assert (
            remaining_tokens > 0
        ), f"remaining tokens must be positive. Given {remaining_tokens=}. Increase max_tokens or lower reasoning_budget."

        # 2. append reasoning content to messages and call completion
        messages.append({"role": "assistant", "content": reasoning_content})
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            continue_final_message=True,
        )
        
        response = self.client.completions.create(
            model=model, 
            prompt=prompt, 
            max_tokens=remaining_tokens, 
            **kwargs
        )

        response_data = {
            "reasoning_content": reasoning_content.strip().strip("</think>").strip(),
            "content": response.choices[0].text,
            "finish_reason": response.choices[0].finish_reason,
        }
        return response_data

In [10]:
# Client
model_id = "nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16" # set this to the model you loaded

client = ThinkingBudgetClient(
    base_url="http://0.0.0.0:8000/v1",
    api_key="null",
    tokenizer_name_or_path=model_id
)

In [12]:
resp = client.chat_completion(
    model=model_id,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1,
    max_tokens=512,
    reasoning_budget=128
)
print("Reasoning:", resp["reasoning_content"], "\nContent:", resp["content"])

Reasoning: We need to write a haiku about GPUs. A haiku is 5-7-5 syllable structure. Provide 3 lines, 5 syllables, 7 syllables, 5 syllables. Might mention graphics, processing, cores, etc. Provide a nice poetic haiku. Ensure correct syllable count.

Let's craft: "Silicon veins pulse / Parallel rivers compute dreams / Light builds worlds anew"

Check syllables:

Line1: "Silicon veins pulse" -> Sil-i-con (3) veins (1) pulse (1) = 5? Actually "Silicon" is 3 syllables (sil-i-con. 
Content: 
Silicon veins pulse  
Parallel rivers compute dreams  
Light builds worlds anew


## Cleanup and shutdown

To tear down this TensorRT-LLM workflow:

1. In the terminal running `trtllm-serve`, press `Ctrl+C` to stop the server.
2. In the Docker shell, run `exit` to stop the container (`--rm` removes it automatically).
3. Optionally run the next cell to clear notebook-side CUDA cache before your next run.

In [ ]:
import gc
import torch

# Optional notebook-side cleanup (server/container are stopped from terminal)
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Notebook-side CUDA cache cleanup complete.")